# 11 - Account A: mesh benchmark
5 mesh models x 20 subjects x 5 conditions (original, face-64, face-32, face-16, compound), the real-world set (original only), the ViT-S oracle diagnostic, raw projection on the ViT-S meshes, a GLB for every mesh, and the cost table. Everything is written under `/kaggle/working/out_A` and zipped. The run is resumable: re-running skips fits that already succeeded.

In [ ]:
REPO_URL = "https://github.com/<USER>/TrueFace3D.git"   # <- set before running
import os, subprocess, sys
REPO = "/kaggle/working/TrueFace3D"
if os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
print("commit", subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
os.chdir(REPO); sys.path.insert(0, REPO)

In [ ]:
from src.kaggle import setup
setup.main([])

In [ ]:
from pathlib import Path
from src.kaggle import runner, mesh_jobs
OUT = Path('/kaggle/working/out_A'); OUT.mkdir(exist_ok=True)
sets = mesh_jobs.case_sets()
print({k: len(v) for k, v in sets.items()})
jobs = mesh_jobs.matrix_jobs(OUT, sets)
print(len(jobs), 'matrix jobs')

### Phase 1: models x conditions on both GPUs
Two worker processes, one per T4 (`CUDA_VISIBLE_DEVICES`). Logs are in `out_A/logs`.

In [ ]:
codes = runner.launch('src.kaggle.mesh_jobs:run_job', jobs, OUT / 'logs', cwd=REPO)
print('exit codes', codes)

### Phase 2: ViT-S oracle
Fit each degraded image again while substituting the clean image's priors: correspondence only, depth only, or both. This shows how much of the error could be removed by perfect priors.

In [ ]:
oracle = mesh_jobs.oracle_jobs(OUT, sets)
print(len(oracle), 'oracle jobs')
codes = runner.launch('src.kaggle.mesh_jobs:run_job', oracle, OUT / 'logs_oracle', cwd=REPO)
print('exit codes', codes)

### Phase 3: raw projection, GLBs and cost

In [ ]:
from src.kaggle import appearance_jobs, export, cost
tagged = mesh_jobs.collect_records(OUT)
raw = appearance_jobs.raw_projection_from_vits(tagged, OUT / 'appearance')
appearance_jobs.score_pairs(OUT / 'appearance', methods=('raw_projection',))
index = export.export_glbs(tagged, OUT / 'glb')
print(len(raw), 'raw projections;', len(index), 'GLB entries')

In [ ]:
import json
rows = cost.cost_rows(tagged)
vit = next((Path(r.case_dir) / 'fit' / 'priors.npz' for _, r in tagged if r.model_id == 'vits' and r.fit_status == 'success'), None)
image = next((c.image_path for c in sets['clean']), None)
profiles = cost.profile_models(OUT / 'flops', vit, image) if vit else {}
summary = cost.summarize_cost(rows, profiles)
(OUT / 'cost_rows.json').write_text(json.dumps(rows, indent=1, default=str))
(OUT / 'cost_summary.json').write_text(json.dumps(summary, indent=1, default=str))
from src.reporting import table_markdown
from IPython.display import Markdown
Markdown(table_markdown(summary))

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/out_A', 'zip', OUT)
print('zip written')